In [ ]:
import json

with open("../data/sheet_smart_with_failure_with_exception_chain_with_baseline_patch_check_with_tokens.json", "r") as f:
    data = json.load(f)

# Report below_10k_exact, below_10k_class, below_10k_method, below_10k_total, and same for above_10k

above_10k_exceptions:dict = {}
above_10k_assertion_exceptions:dict = {}
below_10k_exceptions:dict = {}
below_10k_exception_assertions:dict = {}

for props in data:  # list entries
    cc1 = (props.get("Consistency check #1") or {}).get("Included (patch exists within calltree)")
    calls = (props.get("Metadata") or {}).get("Calls")
    bpc = props.get("BaselinePatchCheck") or {}
    clazz = bpc.get("class")
    method = bpc.get("method")
    exact = clazz and method
    experiment = props.get("Nr")  # or props.get("Graph"), etc.
    exception_error = not props.get("is_assertion_failure")
    isAbove10k = calls > 10000
    pathEdges = props.get("ExceptionChain").get("path_edges")

    # If consistency check #1 is false, skip
    if cc1 is False:
        continue
    if isAbove10k:
        if exception_error:
            # Store experiment, class, method for above 10k exceptions
            above_10k_exceptions[experiment] = (clazz, method, pathEdges)
        else:
            # Store experiment, class, method for above 10k assertion failures
            above_10k_assertion_exceptions[experiment] = (clazz, method, pathEdges)
    else:
        if exception_error:
            # Store experiment, class, method for below 10k exceptions
            below_10k_exceptions[experiment] = (clazz, method, pathEdges)
        else:
            # Store experiment, class, method for below 10k assertion failures
            below_10k_exception_assertions[experiment] = (clazz, method, pathEdges)

# Now we can report the counts and details
print("Above 10k Exceptions:")
print(f"Total: {len(above_10k_exceptions)}")
for exp, (clazz, method, pathEdges) in above_10k_exceptions.items():
    print(f"Experiment: {exp}, Class: {clazz}, Method: {method}, Path Edges: {pathEdges}")
print("\nAbove 10k Assertion Failures:")
print(f"Total: {len(above_10k_assertion_exceptions)}")
for exp, (clazz, method, pathEdges) in above_10k_assertion_exceptions.items():
    print(f"Experiment: {exp}, Class: {clazz}, Method: {method}, Path Edges: {pathEdges}")
print("\nBelow 10k Exceptions:")
print(f"Total: {len(below_10k_exceptions)}")
for exp, (clazz, method, pathEdges) in below_10k_exceptions.items():
    print(f"Experiment: {exp}, Class: {clazz}, Method: {method}, Path Edges: {pathEdges}")
print("\nBelow 10k Assertion Failures:")
print(f"Total: {len(below_10k_exception_assertions)}")
for exp, (clazz, method, pathEdges) in below_10k_exception_assertions.items():
    print(f"Experiment: {exp}, Class: {clazz}, Method: {method}, Path Edges: {pathEdges}")


Above 10k Exceptions:
Total: 8
Experiment: 6, Class: True, Method: True, Path Edges: 8
Experiment: 8, Class: False, Method: False, Path Edges: 2
Experiment: 13, Class: True, Method: True, Path Edges: 2
Experiment: 57, Class: False, Method: False, Path Edges: 5
Experiment: 60, Class: True, Method: True, Path Edges: 5
Experiment: 77, Class: False, Method: False, Path Edges: 2
Experiment: 93, Class: False, Method: False, Path Edges: 7
Experiment: 99, Class: False, Method: False, Path Edges: 94

Above 10k Assertion Failures:
Total: 33
Experiment: 2, Class: True, Method: False, Path Edges: 7
Experiment: 3, Class: False, Method: False, Path Edges: 1
Experiment: 12, Class: False, Method: False, Path Edges: 1
Experiment: 16, Class: False, Method: False, Path Edges: 5
Experiment: 18, Class: False, Method: False, Path Edges: 14
Experiment: 19, Class: False, Method: False, Path Edges: 1
Experiment: 23, Class: False, Method: False, Path Edges: 14
Experiment: 27, Class: False, Method: False, Path E

In [2]:
# Calculate the min, 1st quartile, median, 3rd quartile, and max of the number of path edges for above 10k exceptions and assertion failures, and the same for below 10k
def calculate_statistics(path_edges_dict):
  path_edges_counts = []
  for _, (_, _, edges) in path_edges_dict.items():
    # edges may already be an int (path edge count) or a sequence; handle both
    if isinstance(edges, int):
      cnt = edges
    else:
      cnt = len(edges) if edges is not None else 0
    path_edges_counts.append(cnt)

  if not path_edges_counts:
    return None  # No data to calculate statistics
  path_edges_counts.sort()
  n = len(path_edges_counts)
  min_val = path_edges_counts[0]
  q1 = path_edges_counts[n // 4] if n >= 4 else min_val
  median = path_edges_counts[n // 2] if n >= 2 else min_val
  q3 = path_edges_counts[(3 * n) // 4] if n >= 4 else median
  max_val = path_edges_counts[-1]
  return min_val, q1, median, q3, max_val

print("\nAbove 10k Exceptions Path Edges Statistics:")
above_10k_exception_stats = calculate_statistics(above_10k_exceptions)
if above_10k_exception_stats:
  print(f"Min: {above_10k_exception_stats[0]}, Q1: {above_10k_exception_stats[1]}, Median: {above_10k_exception_stats[2]}, Q3: {above_10k_exception_stats[3]}, Max: {above_10k_exception_stats[4]}")

print("\nAbove 10k Assertion Failures Path Edges Statistics:")
above_10k_assertion_stats = calculate_statistics(above_10k_assertion_exceptions)
if above_10k_assertion_stats:
  print(f"Min: {above_10k_assertion_stats[0]}, Q1: {above_10k_assertion_stats[1]}, Median: {above_10k_assertion_stats[2]}, Q3: {above_10k_assertion_stats[3]}, Max: {above_10k_assertion_stats[4]}")

print("\nBelow 10k Exceptions Path Edges Statistics:")
below_10k_exception_stats = calculate_statistics(below_10k_exceptions)
if below_10k_exception_stats:
  print(f"Min: {below_10k_exception_stats[0]}, Q1: {below_10k_exception_stats[1]}, Median: {below_10k_exception_stats[2]}, Q3: {below_10k_exception_stats[3]}, Max: {below_10k_exception_stats[4]}")

print("\nBelow 10k Assertion Failures Path Edges Statistics:")
below_10k_assertion_stats = calculate_statistics(below_10k_exception_assertions)
if below_10k_assertion_stats:
  print(f"Min: {below_10k_assertion_stats[0]}, Q1: {below_10k_assertion_stats[1]}, Median: {below_10k_assertion_stats[2]}, Q3: {below_10k_assertion_stats[3]}, Max: {below_10k_assertion_stats[4]}")


Above 10k Exceptions Path Edges Statistics:
Min: 2, Q1: 2, Median: 5, Q3: 8, Max: 94

Above 10k Assertion Failures Path Edges Statistics:
Min: 1, Q1: 1, Median: 3, Q3: 5, Max: 19

Below 10k Exceptions Path Edges Statistics:
Min: 1, Q1: 3, Median: 4, Q3: 6, Max: 10

Below 10k Assertion Failures Path Edges Statistics:
Min: 1, Q1: 1, Median: 1, Q3: 3, Max: 10
